# 02 · Results — test set

Run after training:

```bash
python -m src.evaluate          # test set, once
python -m src.report
python -m src.export --benchmark
```
This notebook reads those outputs and adds a patient-level view.

In [ ]:
import os, sys, json
os.chdir('..') if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, '.')
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
SPLIT = 'test'
OUT = f'reports/eval/{SPLIT}'
df = pd.read_csv(f'{OUT}/per_patient.csv')
display(Markdown(open(f'{OUT}/results.md').read()))

## Per-patient Dice by model

In [ ]:
Image(f'{OUT}/model_comparison.png')

## Missing-sequence test
Each sequence removed in turn, for the ResNet34 U-Net trained with and without modality dropout.

In [ ]:
Image(f'{OUT}/missing_sequences.png')

## Best and worst patients
The reasons listed in the results above are derived from each patient's metrics (tumor size, grade, distance errors, volume ratio).

In [ ]:
Image(f'{OUT}/gallery.png')

## Does the uncertainty map point at the errors?
AUROC = how well per-voxel uncertainty separates wrong from right voxels inside the brain (0.5 = no better than chance).

In [ ]:
display(Image(f'{OUT}/uncertainty.png'))
df[df['drop'].isna()].groupby(['model', 'tta'])[['unc_auroc_WT', 'err_top1pct_WT', 'err_rest_WT']].mean().round(3)

## Volume agreement (mL)
Voxels are 1 mm³, so predicted volumes can be compared directly with the expert's.

In [ ]:
best = json.load(open(f'{OUT}/best_model.json'))['best']
d = df[(df.model == best) & ~df.tta.astype(bool) & df['drop'].isna()]
d[[f'vol_err_{r}' for r in ['WT', 'TC', 'ET']]].describe().round(1)

## Patient-level view
Tumor area per slice and a 3D view for one test patient (the same views as the app).

In [ ]:
from src.brats_io import BraTSSource, check_and_load
from src.inference import TorchPredictor, load_model, predict_volume, to_regions
from src.utils import load_config, get_device
from src import viz
cfg = load_config(); dev = get_device()
pred = TorchPredictor(load_model(f'checkpoints/{best}/best.pt', dev)[0], dev)
pid = d.sort_values('dice_WT').iloc[len(d) // 2].pid      # a typical patient
vol, seg, _ = check_and_load(BraTSSource(cfg['raw_path']), pid)
res = predict_volume(pred, vol, cfg)
reg = to_regions(res['probs'].astype(np.float32))
print(pid, {r: f'{reg[i].sum() / 1000:.1f} mL' for i, r in enumerate(['WT', 'TC', 'ET'])})
viz.area_chart(reg).show()
viz.view_3d(reg, res['brain']).show()

## Speed (ONNX export)

In [ ]:
json.load(open('reports/benchmark.json'))